# Flujo completo: DXF a columnas discretizadas

Este flujo está actualizado para la API nueva:

- `x_positions` se asocia a una superficie de falla específica.
- `small_area_scale` se define respecto al área total del DXF, no respecto al polígono mayor.
- Los estratos no identificados deben resolverse antes de construir configuraciones.
- La calibración GQ/H + MRDF se ejecuta por estrato discretizado cuando `calibrate=True`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "dynaengine").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynaengine import (
    CalibrationSettings,
    apply_material_aliases,
    extract_columns_from_dxf,
    prepare_column_configs,
    process_column_config,
    process_dxf_folder,
    resolve_unidentified_materials,
    plot_dxf_extraction,
    plot_raw_column,
    plot_discretized_column,
    plot_column_discretized_detailed,
    summarize_polygon_areas,
)

## 1. Definición de materiales

In [ ]:
MATERIALS = [
    {
        "material_name": "Material de poza",
        "unit_weight_kn_m3": 19,
        "shear_velocity": {"depth": [0, 5, 10, 15], "vs": [300, 350, 440, 550]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "darendeli_2001",
            "sigma_vertical": 100,
            "soil_parameters": {"IP": 0.0, "OCR": 1.0, "k0": 0.7, "frequency": 1.0, "N": 10},
        },
    },
    {
        "material_name": "Material del dique",
        "unit_weight_kn_m3": 20,
        "shear_velocity": {"depth": [0, 8, 15, 20], "vs": [320, 380, 420, 550]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "menq_2003",
            "sigma_vertical": 100,
            "soil_parameters": {"Cu": 18.0, "D50": 8.0, "k0": 0.7, "N": 10},
        },
    },
    {
        "material_name": "Grava arcillosa",
        "unit_weight_kn_m3": 19,
        "shear_velocity": {"depth": [0, 10, 20, 25], "vs": [200, 330, 540, 600]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "wang_2021",
            "sigma_vertical": 100,
            "soil_parameters": {"soil_group": "Nonplastic silty sand group", "Cu": 2.0, "CF": 50.0, "e": 0.7, "D50": 1.0, "wc": 1.0, "k0": 0.7},
        },
    },
    {
        "material_name": "Grava arenosa",
        "unit_weight_kn_m3": 19,
        "shear_velocity": {"depth": [0, 24, 30, 40], "vs": [230, 300, 440, 700]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "rollins_2020",
            "sigma_vertical": 100,
            "soil_parameters": {"Cu": 1.0, "k0": 1.0},
        },
    },
    {
        "material_name": "Grava pobremente gradada",
        "unit_weight_kn_m3": 19.0,
        "shear_velocity": {"depth": [0, 24, 30, 35], "vs": [230, 300, 440, 500]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "ishibashi_1993",
            "sigma_vertical": 100,
            "soil_parameters": {"IP": 50.0, "k0": 0.5},
        },
    },
    {
        "material_name": "Estrato no identificado 1",
        "unit_weight_kn_m3": 19.0,
        "shear_velocity": {"depth": [0, 24, 30, 35], "vs": [230, 300, 440, 550]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "rojas_2019",
            "sigma_vertical": 100,
            "soil_parameters": {"k0": 1.0},
        },
    },
]

## 2. Parámetros del DXF

In [ ]:
dxf_path = ROOT / "examples" / "data" / "section_01.dxf"

# Las posiciones X ahora se asocian explícitamente a una superficie de falla.
# Use llaves "failure_1", "failure_2", ... o llaves numéricas 1, 2, ...
X_POSITIONS_BY_FAILURE = {
    "failure_1": [250.0, 480.0],
}

# Umbral opcional de área pequeña: area_poligono / area_total_DXF.
SMALL_AREA_SCALE = 0.01

# Alias opcionales para nombres de material ya identificados en el DXF.
MATERIAL_ALIASES = {
    # "Nombre en DXF": "Nombre caracterizado en MATERIALS",
}

# Acciones para estratos con nombre "Estrato no identificado ...".
# Debe usarse una de dos opciones por estrato no identificado:
# 1) asignar a un material ya caracterizado; o
# 2) caracterizarlo como un material nuevo, opcionalmente con otro nombre.
UNIDENTIFIED_MATERIAL_ACTIONS = {
    # Opción 1: asignar a material caracterizado
    # "Estrato no identificado 1": {"action": "assign", "material": "Grava arenosa"},

    # Opción 2: caracterizar el estrato no identificado como material nuevo
    # "Estrato no identificado 2": {
    #     "action": "characterize",
    #     "name": "Material nuevo",
    #     "properties": {
    #         "unit_weight_kn_m3": 19,
    #         "shear_velocity": {"depth": [0, 10, 20], "vs": [250, 320, 420]},
    #         "shear_properties": {"c": 0, "phi": 34},
    #         "dynamic_model": {
    #             "model_type": "darendeli_2001",
    #             "sigma_vertical": 100,
    #             "soil_parameters": {"IP": 5, "OCR": 1, "k0": 0.7, "frequency": 1, "N": 10},
    #         },
    #     },
    # },
}

## 3. Lectura del DXF y resolución de materiales

In [ ]:
if not dxf_path.exists():
    raise FileNotFoundError(
        f"No existe el DXF de ejemplo: {dxf_path}. Ajuste dxf_path antes de continuar."
    )

extraction = extract_columns_from_dxf(
    dxf_path,
    x_positions=X_POSITIONS_BY_FAILURE,
    material_aliases=MATERIAL_ALIASES,
    failure_types={"failure_1": "tipo_de_falla"},
    small_area_scale=SMALL_AREA_SCALE,
)

materials_for_dxf, unidentified_aliases = resolve_unidentified_materials(
    MATERIALS,
    extraction.unidentified_materials,
    actions=UNIDENTIFIED_MATERIAL_ACTIONS,
    material_aliases=MATERIAL_ALIASES,
)

ALL_MATERIAL_ALIASES = {**MATERIAL_ALIASES, **unidentified_aliases}
columns = apply_material_aliases(extraction.columns, ALL_MATERIAL_ALIASES)

print(f"Materiales encontrados en DXF: {extraction.material_names}")
print(f"Estratos no identificados: {extraction.unidentified_materials}")
print(f"Alias aplicados: {ALL_MATERIAL_ALIASES}")
print(f"Columnas extraídas: {list(columns.keys())}")

pd.DataFrame(extraction.polygon_area_summary).head()


depth_reference = pd.DataFrame([
    {
        "column_name": name,
        "x_position": data.get("x_position"),
        "failure_surface": data.get("failure_surface"),
        "failure_type": data.get("failure_type"),
        "external_elevation_m": data.get("external_elevation"),
        "freatic_depth_m": data.get("freatic"),
        "failure_depth_m": data.get("depth_failure_surface"),
    }
    for name, data in columns.items()
])
display(depth_reference)

## 4. Visualización y revisión de áreas pequeñas

La tabla reporta `area_ratio_to_total`. Las áreas con `is_small_area=True` se omiten al cortar columnas y el intervalo vertical se reparte hacia las capas adyacentes hasta el centro del área pequeña.

In [ ]:
from dynaengine.dxf import _read_dxf_layers, _generate_clean_polygons

external, freatic, material, failure, text = _read_dxf_layers(dxf_path)
clean_polygons, total_area = _generate_clean_polygons(
    external,
    material,
    text,
    small_area_scale=SMALL_AREA_SCALE,
)

area_summary = pd.DataFrame(
    summarize_polygon_areas(clean_polygons, small_area_scale=SMALL_AREA_SCALE)
)
print(f"Área total del DXF: {total_area:.3f}")
display(area_summary)

small_areas = area_summary[area_summary["is_small_area"]] if not area_summary.empty else area_summary
if small_areas.empty:
    print(f"No se detectaron áreas pequeñas con area/area_total < {SMALL_AREA_SCALE:.3f}.")
else:
    display(small_areas)

plot_x_positions = [x for positions in X_POSITIONS_BY_FAILURE.values() for x in positions]
fig, ax = plot_dxf_extraction(
    clean_polygons,
    x_positions=plot_x_positions,
    highlight_small_areas=True,
    small_area_scale=SMALL_AREA_SCALE,
    annotate_areas=True,
)
plt.show()

## 5. Preparación de configuraciones de columna

In [ ]:
configs = prepare_column_configs(
    columns,
    materials_for_dxf,
    target_frequency_hz=25,
)

print(f"Configuraciones preparadas para: {list(configs.keys())}")

## 6. Procesamiento de primera columna

In [ ]:
first_name = next(iter(configs))
result = process_column_config(configs[first_name], calibrate=False)

print(f"Columna procesada: {first_name}")
print(f"Capas no discretizadas: {len(result.raw)}")
print(f"Segmentos discretizados: {len(result.discretized)}")
print(f"natural_frequency_hz presente: {'natural_frequency_hz' in result.discretized.columns}")

### 6.1 Datos de columna no discretizada

In [ ]:
result.raw[["layer_id", "material_name", "top_m", "bottom_m", "thickness_m", "shear_velocity_m_s"]].head(10)

### 6.2 Gráfico de columna no discretizada

In [ ]:
fig, ax = plot_raw_column(result.raw)
plt.show()

### 6.3 Datos de columna discretizada

In [ ]:
result.discretized[[
    "segment_id", "material_name", "top_m", "bottom_m", "thickness_m",
    "shear_velocity_m_s", "natural_frequency_hz",
    "failure_surface_name", "passes_failure_surface",
]].head(15)

### 6.4 Gráfico simple de columna discretizada

In [ ]:
fig, ax = plot_discretized_column(result.discretized)
plt.show()

### 6.5 Gráfico detallado de columna discretizada

In [ ]:
fig, axes = plot_column_discretized_detailed(result.discretized)
plt.show()

## 7. Procesamiento de segunda columna, si existe

In [ ]:
if len(configs) >= 2:
    second_name = list(configs.keys())[1]
    result2 = process_column_config(configs[second_name], calibrate=False)
    print(f"Columna procesada: {second_name}")
    print(f"Capas no discretizadas: {len(result2.raw)}")
    print(f"Segmentos discretizados: {len(result2.discretized)}")
    fig, axes = plot_column_discretized_detailed(result2.discretized)
    plt.show()
else:
    print("Solo hay una columna configurada.")

## 8. Comparación de perfiles Vs entre columnas

In [ ]:
def segment_profile_xy(disc, value_column):
    x_values = []
    y_values = []
    previous_value = None
    for _, row in disc.iterrows():
        top = row["top_m"]
        bottom = row["bottom_m"]
        value = row[value_column]
        if previous_value is not None:
            x_values.extend([previous_value, value])
            y_values.extend([top, top])
        x_values.extend([value, value])
        y_values.extend([top, bottom])
        previous_value = value
    return x_values, y_values

comparison_results = {first_name: result}
if "result2" in globals():
    comparison_results[second_name] = result2

fig, axes = plt.subplots(1, len(comparison_results), figsize=(6 * len(comparison_results), 8), sharey=True)
if len(comparison_results) == 1:
    axes = [axes]

for ax, (column_name, result_data) in zip(axes, comparison_results.items()):
    disc = result_data.discretized
    max_vs = disc["shear_velocity_m_s"].max()
    x_values, y_values = segment_profile_xy(disc, "shear_velocity_m_s")
    ax.plot(x_values, y_values, linewidth=2)
    ax.set_xlim(0, max_vs * 1.1)
    ax.invert_yaxis()
    ax.set_xlabel("Vs (m/s)")
    ax.set_title(f"Perfil Vs - {column_name}", weight="bold")
    ax.grid(True, axis="x", alpha=0.25)

axes[0].set_ylabel("Profundidad (m)")
fig.suptitle("Comparación de perfiles Vs entre columnas", fontsize=14, weight="bold")
fig.tight_layout()
plt.show()

## 9. Procesamiento de todas las columnas

In [ ]:
all_results = {}
for col_name, config in configs.items():
    processed = process_column_config(config, calibrate=False)
    all_results[col_name] = processed
    print(f"✓ {col_name}: {len(processed.discretized)} segmentos")

print(f"Total de columnas procesadas: {len(all_results)}")

## 10. Resumen comparativo

In [ ]:
summary_data = []
for col_name, processed in all_results.items():
    raw_df = processed.raw
    disc_df = processed.discretized
    summary_data.append({
        "Columna": col_name,
        "Capas raw": len(raw_df),
        "Segmentos discretizados": len(disc_df),
        "Profundidad máxima (m)": raw_df["bottom_m"].max(),
        "Vs promedio (m/s)": raw_df["shear_velocity_m_s"].mean(),
        "Peso unitario promedio (kN/m³)": raw_df["unit_weight_kn_m3"].mean(),
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

## 11. Distribución de materiales

In [ ]:
material_summary = []
for col_name, processed in all_results.items():
    raw_df = processed.raw
    for material_name in raw_df["material_name"].unique():
        material_data = raw_df[raw_df["material_name"] == material_name]
        material_summary.append({
            "Columna": col_name,
            "Material": material_name,
            "Número de capas": len(material_data),
            "Espesor total (m)": material_data["thickness_m"].sum(),
            "Profundidad superior (m)": material_data["top_m"].min(),
            "Profundidad inferior (m)": material_data["bottom_m"].max(),
            "Vs promedio (m/s)": material_data["shear_velocity_m_s"].mean(),
        })

material_df = pd.DataFrame(material_summary)
display(material_df)

## 12. Calibración opcional por estrato discretizado

Al usar `calibrate=True`, cada fila de la columna discretizada se calibra de forma independiente. El resultado incluye `theta_1` a `theta_5`, `P1` a `P3`, `dmin`, `gqh_gamma_ref`, `gqh_score` y `mrdf_score`.

In [ ]:
CALIBRATION_SETTINGS = CalibrationSettings(
    gqh_init_points=2,
    gqh_iterations=3,
    mrdf_init_points=2,
    mrdf_iterations=3,
    random_state=1,
)

try:
    calibrated_result = process_column_config(
        configs[first_name],
        calibrate=True,
        calibration_settings=CALIBRATION_SETTINGS,
    )
    display(calibrated_result.calibrated[[
        "segment_id", "material_name", "top_m", "bottom_m",
        "theta_1", "theta_2", "theta_3", "theta_4", "theta_5",
        "P1", "P2", "P3", "dmin", "gqh_gamma_ref", "gqh_score", "mrdf_score",
    ]].head())
except ImportError as exc:
    print("Para ejecutar calibración instale la dependencia opcional: bayesian-optimization")
    print(exc)

## 13. Procesamiento por carpeta

In [ ]:
# Ejemplo batch con la API actualizada. Descomente para procesar todos los DXF de una carpeta.
# batch_results = process_dxf_folder(
#     section_folder=ROOT / "examples" / "data",
#     x_positions_by_file={"section_01": X_POSITIONS_BY_FAILURE},
#     materials=MATERIALS,
#     target_frequency_hz=25,
#     calibrate=False,
#     failure_types_by_file={"section_01": {"failure_1": "tipo_de_falla"}},
#     material_aliases_by_file={"section_01": MATERIAL_ALIASES},
#     unidentified_material_actions_by_file={"section_01": UNIDENTIFIED_MATERIAL_ACTIONS},
#     small_area_scale=SMALL_AREA_SCALE,
#     output_dir=ROOT / "examples" / "output",
# )

## 14. Exportar resultados

In [ ]:
output_dir = ROOT / "examples" / "output"
output_dir.mkdir(parents=True, exist_ok=True)

for col_name, processed in all_results.items():
    raw_file = output_dir / f"{col_name}_raw.csv"
    disc_file = output_dir / f"{col_name}_discretized.csv"
    processed.raw.to_csv(raw_file, index=False)
    processed.discretized.to_csv(disc_file, index=False)
    print(f"✓ Guardado: {raw_file}")
    print(f"✓ Guardado: {disc_file}")

summary_df.to_csv(output_dir / "columns_summary.csv", index=False)
material_df.to_csv(output_dir / "materials_summary.csv", index=False)
print(f"Resúmenes guardados en: {output_dir}")